## Dataset up 

In [ ]:
include("main_utils.jl")
include("data_setup.jl")
include("comix_uk_time_series.jl")
include("bnb_utils.jl")

default_plot_setting()

### Variable tables (data dictionary)

The raw CoMix dump ships a data dictionary (`dd_v1238_20230328.xlsx`) whose
`final_contact` / `final_part` sheets document every column. `country`
(`final_part`, row 1) is the *"Country 2-letter abbreviation"* — the field we
filter on to extract the UK panel below.

In [ ]:
# Variable tables (data dictionary). The `final_contact` / `final_part` sheets
# document every column; `country` (final_part row 1) is the
# "Country 2-letter abbreviation" used for the UK filter below.
dd_contact = DataFrame(XLSX.readtable("../dt_comix_no_public/dd_v1238_20230328.xlsx", "final_contact"))
dd_part    = DataFrame(XLSX.readtable("../dt_comix_no_public/dd_v1238_20230328.xlsx", "final_part"))
@info "Variable tables" contact_vars = nrow(dd_contact) part_vars = nrow(dd_part)
first(dd_part, 5)

### Filter CoMix → CoMix-UK

The raw `contacts.csv` / `part.csv` pool all 20 CoMix countries. Subset each to
`country == "uk"` and write the `_uk.csv` files that the Arrow step (next)
converts. Guarded by `isfile`, so re-runs are a no-op.

In [ ]:
# Subset the raw CoMix CSVs to the UK panel → `_uk.csv` (skipped if present).
# `contacts.csv` / `part.csv` pool all 20 CoMix countries; keep `country == "uk"`.
let dir = "../dt_comix_no_public"
    for name in ("contacts", "part")
        raw = joinpath(dir, name * ".csv")
        uk  = joinpath(dir, name * "_uk.csv")
        if isfile(uk)
            @info "UK CSV already exists, skipping" uk
        else
            @info "Filtering raw CSV → UK" raw uk
            df    = CSV.read(raw, DataFrame)
            df_uk = @subset(df, :country .== "uk")
            @info "  rows" total = nrow(df) uk = nrow(df_uk)
            CSV.write(uk, df_uk)
            @info "  ✓ wrote" uk
        end
    end
end

### Convert CoMix-UK CSVs → Arrow

One-time conversion of `dt_comix_no_public/{contacts_uk,part_uk}.csv` into the
`.arrow` files that `read_comix_uk_raw_contacts_and_part()` (and other readers)
load via `read_arrow_df`. Guarded by `isfile`, so re-runs are a no-op.

In [ ]:
# Convert CoMix-UK source CSVs → Arrow (one-time; skipped if already present).
# `read_comix_uk_raw_contacts_and_part()` and other readers memory-map these
# `.arrow` files via `read_arrow_df` (data_setup.jl).
let dir = "../dt_comix_no_public"
    for name in ("contacts_uk", "part_uk")
        csv   = joinpath(dir, name * ".csv")
        arrow = joinpath(dir, name * ".arrow")
        if isfile(arrow)
            @info "Arrow already exists, skipping" arrow
        else
            @info "Converting CSV → Arrow" csv arrow
            Arrow.write(arrow, CSV.read(csv, DataFrame))
            @info "  ✓ wrote" arrow
        end
    end
end

In [ ]:
df, df_part = read_comix_uk_raw_contacts_and_part();
# Annotate both tables with inc2prev-aligned 7-day weeks (Sunday-start, anchored
# 2021-03-21) so `:mid_date` (the Wednesday label used as the df_dds key below)
# lands 1:1 on the inc2prev / framework week grid.
df, df_part = add_inc2prev_week_chunks!(df, df_part);
#df, df_part = read_adult_chunks(df, df_part);

In [ ]:
df_dds = create_df_dds_chunk(df, df_part)
CSV.write("../dt_intermediate/df_dds.csv", df_dds)

In [ ]:
df_dds = create_df_dds_chunk_by_settings(df, df_part)
CSV.write("../dt_intermediate/df_dds_settings.csv", df_dds)